# 🦥 Scenario D: Molmo-7B with Unsloth

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matheus-rech/pitvqa-surgical-workflow/blob/main/notebooks/06_unsloth_molmo_training.ipynb)

This notebook uses **Unsloth** for efficient fine-tuning of Molmo-7B on PitVQA surgical data.

**Benefits of Unsloth:**
- 🚀 2-5x faster training
- 💾 70% less VRAM
- ✅ Enables 7B+ models on single GPU

**Requirements:**
- Colab Pro (A100 GPU recommended)
- HuggingFace token with write access

---

In [ ]:
#@title 1. Install Unsloth and Dependencies
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir huggingface_hub datasets transformers trl peft accelerate bitsandbytes
!pip install pillow tqdm

print("✅ Dependencies installed!")

In [ ]:
#@title 2. Configuration
import os
import torch

# === CONFIGURATION ===
HF_TOKEN = ""  #@param {type:"string"}
HF_USERNAME = "mmrech"  #@param {type:"string"}

# Model settings
BASE_MODEL = "allenai/Molmo-7B-D-0924"  #@param ["allenai/Molmo-7B-D-0924", "allenai/Molmo-7B-O-0924"]
OUTPUT_MODEL = f"{HF_USERNAME}/pitvqa-molmo-unsloth"

# Dataset
DATASET_NAME = f"{HF_USERNAME}/pitvqa-sage-sft"

# Training settings
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 32
EPOCHS = 3
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2e-5

# Login
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    login()

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("❌ No GPU detected! Enable GPU in Runtime > Change runtime type")

print(f"\n📦 Base model: {BASE_MODEL}")
print(f"📤 Output: {OUTPUT_MODEL}")
print(f"📊 Dataset: {DATASET_NAME}")

In [ ]:
#@title 3. Load Model with Unsloth (4-bit Quantization)
from unsloth import FastLanguageModel

print(f"🔄 Loading {BASE_MODEL} with Unsloth...")
print("   This may take a few minutes...\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect
    load_in_4bit=True,  # 4-bit quantization for memory efficiency
)

print(f"✅ Model loaded!")
print(f"   Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
#@title 4. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth optimized
    random_state=42,
)

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA adapters added!")
print(f"   Trainable: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")
print(f"   Total: {total_params:,}")

In [ ]:
#@title 5. Load and Prepare Dataset
from datasets import load_dataset

print(f"📥 Loading dataset: {DATASET_NAME}")
dataset = load_dataset(DATASET_NAME)

print(f"\n📊 Dataset splits:")
for split, data in dataset.items():
    print(f"   {split}: {len(data)} samples")

# Check columns
print(f"\n📋 Columns: {dataset['train'].column_names}")

In [ ]:
#@title 6. Format Dataset for Training

def format_conversation(example):
    """Convert messages format to training format."""
    messages = example['messages']
    
    # Build conversation string
    conversation = ""
    for msg in messages:
        role = msg['role']
        content = msg['content']
        if role == 'user':
            conversation += f"User: {content}\n"
        elif role == 'assistant':
            conversation += f"Assistant: {content}\n"
    
    return {"text": conversation.strip()}

# Apply formatting
print("🔄 Formatting dataset...")
formatted_dataset = dataset.map(
    format_conversation,
    remove_columns=[col for col in dataset['train'].column_names if col != 'image']
)

print(f"✅ Dataset formatted!")
print(f"\n📝 Sample:")
print(formatted_dataset['train'][0]['text'][:500])

In [ ]:
#@title 7. Configure Training
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./outputs",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="epoch",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    seed=42,
    report_to="none",  # or "wandb" if you want logging
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset['train'],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
)

print("✅ Trainer configured!")
print(f"\n📊 Training settings:")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Gradient accumulation: {GRADIENT_ACCUMULATION}")
print(f"   Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"   Learning rate: {LEARNING_RATE}")

In [ ]:
#@title 8. Train! 🚀
import time

print("="*60)
print("🚀 STARTING TRAINING")
print("="*60)
print(f"\nThis will take approximately 2-4 hours on A100...\n")

start_time = time.time()

# Train
trainer_stats = trainer.train()

elapsed = time.time() - start_time
print(f"\n" + "="*60)
print(f"✅ TRAINING COMPLETE!")
print(f"="*60)
print(f"   Time: {elapsed/3600:.2f} hours")
print(f"   Final loss: {trainer_stats.training_loss:.4f}")

In [ ]:
#@title 9. Save and Push to Hub
from huggingface_hub import create_repo

print(f"📤 Saving model to {OUTPUT_MODEL}...\n")

# Create repo
try:
    create_repo(OUTPUT_MODEL, exist_ok=True)
except Exception as e:
    print(f"Note: {e}")

# Save locally first
model.save_pretrained("./pitvqa-molmo-lora")
tokenizer.save_pretrained("./pitvqa-molmo-lora")

# Push to hub
model.push_to_hub(
    OUTPUT_MODEL,
    token=HF_TOKEN if HF_TOKEN else None,
    commit_message="PitVQA surgical fine-tuned Molmo-7B with Unsloth"
)
tokenizer.push_to_hub(
    OUTPUT_MODEL,
    token=HF_TOKEN if HF_TOKEN else None,
)

print(f"\n🎉 SUCCESS!")
print(f"📦 Model: https://huggingface.co/{OUTPUT_MODEL}")

In [ ]:
#@title 10. (Optional) Merge LoRA and Save Full Model
MERGE_AND_SAVE = False  #@param {type:"boolean"}

if MERGE_AND_SAVE:
    print("🔄 Merging LoRA weights into base model...")
    
    merged_model = model.merge_and_unload()
    
    merged_output = f"{OUTPUT_MODEL}-merged"
    merged_model.push_to_hub(
        merged_output,
        token=HF_TOKEN if HF_TOKEN else None,
        commit_message="Merged PitVQA surgical Molmo-7B"
    )
    
    print(f"✅ Merged model: https://huggingface.co/{merged_output}")
else:
    print("ℹ️ Skipping merge. Set MERGE_AND_SAVE=True to create full merged model.")

In [ ]:
#@title 11. Test the Model
print("🧪 Testing the fine-tuned model...\n")

# Enable inference mode
FastLanguageModel.for_inference(model)

# Test prompt
test_prompt = "User: What surgical phase is shown in this image?\nAssistant:"

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=64,
    temperature=0.7,
    do_sample=True,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"📝 Response:\n{response}")

---

## 🎉 Training Complete!

### Output Models

| Model | URL |
|-------|-----|
| LoRA Adapter | `mmrech/pitvqa-molmo-unsloth` |
| Merged (optional) | `mmrech/pitvqa-molmo-unsloth-merged` |

### Next Steps

1. Compare with HF Skills trained models (MedGemma, Qwen2-VL, SAGE)
2. Run evaluation on test set
3. Deploy best model for inference

### Comparison Metrics

```python
# Load and compare models
models = [
    "mmrech/pitvqa-medgemma-surgical",
    "mmrech/pitvqa-qwen2vl-surgical",
    "mmrech/pitvqa-sage-surgical",
    "mmrech/pitvqa-molmo-unsloth",
]
```